In [41]:
import sys
sys.path.append('../')

import torch
import json

from src.model import Transformer, generate_batch
from src.dataset import (
    NMRDataset, 
    load_multiplicity_codes, 
    load_split, 
    collate_fn
)

import sentencepiece as spm

from functools import partial

In [2]:
tokenizer = spm.SentencePieceProcessor('../checkpoints/nmr2struct.model')

In [4]:
model = Transformer(
    multiplicity_vocab_size=len(NMRDataset.MULTIPLICITY2IDX),
    multiplicity_hidden_dim=48,
    spectrum_d_ff=256,
    spectrum_hidden_dim=104,
    intensity_d_ff=256,
    intensity_hidden_dim=104,
    tgt_vocab_size=tokenizer.vocab_size(),
    d_model=256,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    max_seq_length=400,
    dropout=0.2,
    smiles_pad_token=tokenizer.pad_id(),
)
model.load_state_dict(torch.load('../checkpoints/bimodal_256_new_data_full_155.pt', map_location='cpu'))
model.eval()

Transformer(
  (spectrum_embedding): SpectrumEmbedding(
    (spectrum_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (intensity_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (multiplicity_embs): ModuleDict(
      (C_NMR): Embedding(10, 48)
      (H_NMR): Embedding(10, 48)
    )
  )
  (decoder_embedding): Embedding(512, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_carbon_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_featu

In [5]:
multiplicity_codes = load_multiplicity_codes('../data/multiplicity_codes.json')

In [6]:
test_data = load_split(
    path='../data/test_full_50k.jsonl', 
    multiplicity_codes=multiplicity_codes,
    solvent='CDCl3' # Так собраны данные
)

Loading test_full_50k.jsonl: 0it [00:00, ?it/s]

Processing spectra:   0%|          | 0/9954 [00:00<?, ?it/s]

In [51]:
# Для каждой молекулы есть разное количество 1H и 13C спектров
# Датасет устроен так что он каждый раз выдает случайную пару спектров
# И с вероятностью 20% маскирует один из спектров
# Если хочется работать иначе с этим - см. src/dataset.py L153

test_dataset = NMRDataset(
    data=test_data,
    solvent='CDCl3',
    tokenizer=tokenizer,
    smiles_bos_id=tokenizer.bos_id(),
    smiles_eos_id=tokenizer.eos_id(),
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    shuffle=False, 
    batch_size=4, 
    collate_fn=partial(
        collate_fn, 
        smiles_pad_id=tokenizer.pad_id(),
        spectrum_pad_token=-1000.,
        max_smiles_len=400
    )
)

In [52]:
data = next(iter(test_loader))

predictions = generate_batch(
    model=model,
    src_c_nmr=data['C_NMR'],
    src_h_nmr=data['H_NMR'],
    tokenizer=tokenizer,
    max_tokens=400
)

In [62]:
for pred_smiles, true_smiles in zip(predictions, tokenizer.decode(data['smiles'].squeeze().numpy().tolist())):
    print(f"True SMILES: {true_smiles}", f"Predicted SMILES: {pred_smiles}", sep='\t')

True SMILES: O=C1OC(=O)c2ccccc21	Predicted SMILES: O=C1OC(=O)c2ccccc21
True SMILES: Clc1nc(Cl)c2ccccc2n1	Predicted SMILES: O=C(O)c1cccc(C(F)(F)F)c1
True SMILES: c1ccc2c(N3CCCC3)ncnc2c1	Predicted SMILES: c1ccc2c(N3CCCC3)ncnc2c1
True SMILES: COc1ccc(C=O)cc1OC	Predicted SMILES: COc1ccc(C=O)cc1OC
